# 投资组合回测框架使用示例

本 notebook 演示如何使用可扩展的投资组合回测框架

In [1]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np

from portfolio_backtest import (
    BacktestEngine,
    RiskParityStrategy,
    MeanVarianceStrategy
)
from portfolio_backtest.visualization import BacktestVisualizer
from portfolio_backtest.utils import load_price_data, align_columns

## 1. 加载数据

In [2]:
# # 加载价格数据
# price_df = load_price_data('./market_close.csv')
# print(f"数据形状: {price_df.shape}")
# print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
# print(f"\n资产列表:")
# for col in price_df.columns:
#     print(f"  - {col}")

# price_df.head()

In [3]:
# 加载价格数据
price_df = pd.read_excel('./market_close.xlsx')
price_df.columns = price_df.iloc[2]
price_df = price_df.iloc[4:]
price_df['日期'] = pd.to_datetime(price_df['日期'])
price_df = price_df.set_index('日期')
print(f"数据形状: {price_df.shape}")
print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
print(f"\n资产列表:")
for col in price_df.columns:
    print(f"  - {col}")

price_df.head()

数据形状: (2550, 10)
日期范围: 2015-09-01 00:00:00 到 2026-03-06 00:00:00

资产列表:
  - 上证指数
  - 创业板指
  - 纳斯达克指数
  - 道琼斯工业平均
  - 中证转债
  - 中债-商业银行二级资本债券财富(总值)指数
  - 中债-新综合财富(1年以下)指数
  - 中债-新综合财富(1-3年)指数
  - SGE黄金9999
  - ICE布油


2,上证指数,创业板指,纳斯达克指数,道琼斯工业平均,中证转债,中债-商业银行二级资本债券财富(总值)指数,中债-新综合财富(1年以下)指数,中债-新综合财富(1-3年)指数,SGE黄金9999,ICE布油
日期,,,,,,,,,,
2015-09-01,3166.6239,1889.491,29556.128472,102375.19292,291.3647,108.0298,150.7559,161.255,234.6,320.098792
2015-09-02,3160.167,1855.032,30218.897762,104025.844422,289.3253,108.1052,150.7703,161.2732,234.9,331.900323
2015-09-07,3080.4201,1893.521,29782.236928,102385.372992,292.4632,108.0971,150.8428,161.3674,231,314.232128
2015-09-08,3170.4522,2001.156,30622.641327,104957.766252,303.3656,108.0641,150.8641,161.3759,230.58,324.813456
2015-09-09,3243.0889,2071.717,30266.751696,103424.716624,310.0523,107.8927,150.8783,161.3896,231.1,313.260336


## 1.5. 参数设置

In [4]:
# 4个大类，大类之间的风险预算分别为：
# 股票类50%，债券类30%，商品类10%，转债类10%；
# 在股票类中，A股占60%，美股占40%；
# 在债券类中，银行二级资本债券占50%，新综合财富(1年以下)占30%，新综合财富(1-3年)占20%；
# 在商品类中，黄金占50%，原油占50%；
# 转债类占10%，不区分大类。
my_risk_budget = {

    # 共0.5的风险预算分配给股票类，股票类内部按照60%和40%分配
    '上证指数': 0.15,    # 0.5*0.6*0.5
    '创业板指': 0.15,    # 0.5*0.6*0.5
    '纳斯达克指数': 0.1,    # 0.5*0.4*0.5
    '道琼斯工业平均': 0.1,   # 0.5*0.4*0.5

    # 转债类占0.1的风险预算，不区分大类
    '中证转债': 0.1,

    # 共0.3的风险预算分配给债券类，债券类内部按照50%、30%和20%分配
    '中债-商业银行二级资本债券财富(总值)指数': 0.15, # 0.3*0.5
    '中债-新综合财富(1年以下)指数': 0.09,            # 0.3*0.3
    '中债-新综合财富(1-3年)指数': 0.06,              # 0.3*0.2

    # 共0.1的风险预算分配给商品类，商品类内部按照50%和50%分配
    'SGE黄金9999': 0.05,    # 0.1*0.5
    'ICE布油': 0.05
}

if not np.isclose(sum(my_risk_budget.values()), 1.0):
    raise ValueError("风险预算的总和必须为1.0")

# 对应起来的资产列表
my_risk_budget = align_columns(my_risk_budget, price_df.columns)

## 2. 风险平价策略回测

In [5]:
# 创建风险平价策略 - 测试单种方法
rp_strategy = RiskParityStrategy(
    lookback=120,           # 12日回看窗口
    rebalance_freq='ME',    # ME月末调仓, QE 季末调仓
    method='CDD',         # 使用SLSQP优化方法计算权重,也可以选择 'CDD' 方法
    compare_methods=False,   # 关闭方法对比，使用rp_strategy_compare.print_weights_comparison()可以查看权重对比结果
    risk_budget=my_risk_budget
)

# 创建回测引擎
engine = BacktestEngine(
    init_cash=1_000_000,
    freq='1D'
)

# 运行回测
rp_result = engine.run(rp_strategy, price_df)

d:\0信银\26---new\portfolio_backtest\utils\helpers.py:88: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [6]:
# 查看回测统计
print(rp_result.metrics)

rp_result.stats()

{'total_return': np.float64(0.43286896748567755), 'annualized_return': np.float64(0.05283173286407794), 'annualized_volatility': np.float64(0.00935235924935973), 'sharpe_ratio': np.float64(5.509921285170939), 'sortino_ratio': np.float64(9.587489969263707), 'calmar_ratio': np.float64(3.6983007873958775), 'max_drawdown': np.float64(-0.01428540724543903), 'omega_ratio': np.float64(2.301176113002782), 'best_trade': None, 'worst_trade': None, 'win_rate': None}


Start                                  2015-09-01 00:00:00
End                                    2026-03-06 00:00:00
Period                                  2550 days 00:00:00
Start Value                                      1000000.0
End Value                                   1432868.967486
Total Return [%]                                 43.286897
Benchmark Return [%]                            144.795482
Max Gross Exposure [%]                               100.0
Total Fees Paid                                        0.0
Max Drawdown [%]                                  1.428541
Max Drawdown Duration                     67 days 00:00:00
Total Trades                                         11042
Total Closed Trades                                  11032
Total Open Trades                                       10
Open Trade PnL                                 92006.51513
Win Rate [%]                                     83.303118
Best Trade [%]                                   54.6806

In [7]:
rp_result.weights

2,上证指数,创业板指,纳斯达克指数,道琼斯工业平均,中证转债,中债-商业银行二级资本债券财富(总值)指数,中债-新综合财富(1年以下)指数,中债-新综合财富(1-3年)指数,SGE黄金9999,ICE布油
2016-03-31,0.007067,0.004503,0.007796,0.009612,0.008193,0.133442,0.655112,0.160789,0.009994,0.003492
2016-04-29,0.007430,0.004824,0.008912,0.011653,0.008006,0.123978,0.672888,0.147071,0.011205,0.004033
2016-05-31,0.007009,0.004726,0.008667,0.011502,0.007615,0.127937,0.670561,0.145444,0.012255,0.004283
2016-06-30,0.006565,0.004304,0.008640,0.012070,0.008889,0.132178,0.665194,0.144267,0.014492,0.003401
2016-07-29,0.008007,0.004944,0.010163,0.014663,0.011137,0.128174,0.654601,0.155149,0.009992,0.003170
...,...,...,...,...,...,...,...,...,...,...
2025-11-28,0.009495,0.004105,0.004922,0.009645,0.008634,0.162508,0.615126,0.177834,0.003530,0.004200
2025-12-31,0.008904,0.003815,0.004910,0.009485,0.007999,0.155371,0.631248,0.171259,0.002915,0.004094
2026-01-30,0.008391,0.003843,0.004972,0.008213,0.006325,0.159330,0.614830,0.188031,0.002170,0.003895
2026-02-27,0.008584,0.004232,0.005114,0.009492,0.005795,0.171854,0.587433,0.203154,0.001557,0.002785


In [8]:
# 可视化结果
rp_viz = BacktestVisualizer(rp_result)
rp_viz.print_metrics()
rp_viz.plot_summary()


Risk Parity 策略表现
总收益率: 43.29%
年化收益率: 5.28%
年化波动率: 0.94%
夏普比率: 5.510
索提诺比率: 9.587
Calmar比率: 3.698
Omega比率: 2.301
最大回撤: -1.43%



In [9]:
# 权重热力图
rp_viz.plot_weights_heatmap(freq='QE')

In [10]:
# 权重变化分析与调仓点标注
# 分析模型的重要调仓时机
rp_viz.plot_weight_changes_analysis(threshold=0.03)

In [11]:
rp_viz.plot_assets_and_weights(
    price_df,
    asset_name='SGE黄金9999',      # 指定资产名称
    freq='M',               # 按周显示仓位
    weight_alpha=0.3        # 权重柱状图透明度
)

d:\0信银\26---new\portfolio_backtest\visualization\plots.py:419: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [32]:
print(rp_result.weights.columns)
rp_viz.plot_rebalancing_effectiveness(
    price_df,
    asset_name='纳斯达克指数',           # 指定资产
)

Index(['上证指数', '创业板指', '纳斯达克指数', '道琼斯工业平均', '中证转债', '中债-商业银行二级资本债券财富(总值)指数',
       '中债-新综合财富(1年以下)指数', '中债-新综合财富(1-3年)指数', 'SGE黄金9999', 'ICE布油'],
      dtype='object', name=2)


### 如何解读这些图表

**资产走势与权重分析图**：
- **上半部分**：各资产价格走势（归一化到100），帮助理解资产的历史表现
- **下半部分**：对应的权重配置变化，显示模型何时加减仓
- **分析要点**：
  - 当某资产价格下跌时，模型是否增加权重（抄底）或减少权重（止损）
  - 当某资产价格上涨时，模型是否减少权重（获利了结）或增加权重（追涨）
  - 权重变化频率反映策略的调仓灵敏度

**权重变化分析图**：
- 显示所有资产的权重配置时间序列
- 标注重要的调仓点（权重变化超过阈值）
- 统计月度调仓频率，帮助理解策略的活跃度

## 3. 均值方差策略回测

In [13]:
# 创建均值方差策略（最大化夏普比率）
mv_strategy = MeanVarianceStrategy(
    lookback=60,
    rebalance_freq='ME'
)

# 运行回测
mv_result = engine.run(mv_strategy, price_df)

# 可视化
mv_viz = BacktestVisualizer(mv_result)
mv_viz.print_metrics()
mv_viz.plot_summary()

d:\0信银\26---new\portfolio_backtest\utils\helpers.py:88: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`




Mean Variance 策略表现
总收益率: 111.10%
年化收益率: 11.29%
年化波动率: 10.39%
夏普比率: 1.082
索提诺比率: 1.510
Calmar比率: 0.696
Omega比率: 1.182
最大回撤: -16.22%



## 4. 策略对比

In [14]:
# 创建多个策略进行对比
strategies = [
    RiskParityStrategy(lookback=60, rebalance_freq='ME',method='CDD'),
    RiskParityStrategy(lookback=120, rebalance_freq='QE',method='CDD'),
    MeanVarianceStrategy(lookback=60, rebalance_freq='ME'),
]

names = ['风险平价(60日/月)', '风险平价(120日/季)', '均值方差(60日/月)']

# 运行所有策略
results = []
for strategy, name in zip(strategies, names):
    result = engine.run(strategy, price_df)
    results.append(result)
    print(f"{name}: 总收益={result.metrics['total_return']*100:.2f}%, 夏普={result.metrics['sharpe_ratio']:.3f}")

d:\0信银\26---new\portfolio_backtest\utils\helpers.py:88: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



风险平价(60日/月): 总收益=43.05%, 夏普=5.534
风险平价(120日/季): 总收益=43.09%, 夏普=5.866


d:\0信银\26---new\portfolio_backtest\utils\helpers.py:88: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

d:\0信银\26---new\portfolio_backtest\utils\helpers.py:88: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



均值方差(60日/月): 总收益=111.10%, 夏普=1.082


In [15]:
# 累计收益对比图
BacktestVisualizer.compare_results(results, names=names)

In [16]:
# 指标对比表
comparison_table = BacktestVisualizer.compare_metrics_table(results, names)
comparison_table

,总收益率 (%),年化收益率 (%),年化波动率 (%),夏普比率,索提诺比率,Calmar比率,最大回撤 (%)
风险平价(60日/月),43.054750,5.258741,0.926883,5.534447,9.318243,3.249011,-1.618566
风险平价(120日/季),43.094223,5.262898,0.875015,5.866490,10.494076,4.986041,-1.055526
均值方差(60日/月),111.103627,11.287766,10.389123,1.081679,1.510492,0.695775,-16.223289


## 5. 创建自定义策略

继承 `BaseStrategy` 即可创建自己的策略

In [17]:
from portfolio_backtest.strategies.base import BaseStrategy

class EqualWeightStrategy(BaseStrategy):
    """等权重策略 - 自定义策略示例"""
    
    def __init__(self, rebalance_freq='ME'):
        super().__init__(name="Equal Weight", rebalance_freq=rebalance_freq)
        self.rebalance_freq = rebalance_freq
    
    def generate_weights(self, price_df, rebalance_mask=None):
        price_df = self.validate_data(price_df)
        
        if rebalance_mask is None:
            rebalance_dates = self.get_rebalance_dates(price_df, self.rebalance_freq)
            rebalance_mask = pd.Series(
                price_df.index.isin(rebalance_dates),
                index=price_df.index
            )
        
        n_assets = price_df.shape[1]
        equal_weight = 1.0 / n_assets
        
        rebalance_dates = price_df.index[rebalance_mask]
        weights_list = [np.full(n_assets, equal_weight) for _ in rebalance_dates]
        
        return pd.DataFrame(
            weights_list,
            index=rebalance_dates,
            columns=price_df.columns
        )

# 使用自定义策略
ew_strategy = EqualWeightStrategy(rebalance_freq='ME')
ew_result = engine.run(ew_strategy, price_df)

ew_viz = BacktestVisualizer(ew_result)
ew_viz.print_metrics()


Equal Weight 策略表现
总收益率: 145.00%
年化收益率: 13.69%
年化波动率: 10.58%
夏普比率: 1.265
索提诺比率: 1.794
Calmar比率: 0.844
Omega比率: 1.209
最大回撤: -16.22%



d:\0信银\26---new\portfolio_backtest\utils\helpers.py:88: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

